# Reencontra — Consultas de contexto de SP

Execute depois do pipeline. As cinco consultas usam a Ouro derivada das estatísticas reais do MJSP.
1. Totais anuais e cobertura dos campos selecionados.
2. Distribuição mensal.
3. Distribuição por faixa etária.
4. Distribuição por sexo.
5. Variação de registros entre 2023 e 2024.

São registros informados, não uma contagem deduplicada de pessoas. Localizações de um ano podem se referir a desaparecimentos anteriores.
Não subtraia os dois indicadores nem interprete seu quociente como taxa de resolução.
O indicador de cobertura verifica campos preenchidos, sem garantir a qualidade ou a completude da notificação.
As estatísticas não medem efeito causal do Reencontra. A consulta de impacto do MVP continua pendente.


## Consulta 1

Na versão analisada, localização em 2025 tem 179 de 180 combinações válidas. A soma de 16.480 é dos valores disponíveis e está incompleta em um campo.


In [0]:
display(spark.sql("\n    SELECT\n        d.ano,\n        f.indicador,\n        SUM(f.quantidade) AS quantidade_informada,\n        COUNT(*) AS combinacoes_disponiveis,\n        180 AS combinacoes_esperadas,\n        CASE\n            WHEN COUNT(*) = 180 THEN 'Completo nos campos selecionados'\n            ELSE 'Incompleto nos campos selecionados'\n        END AS cobertura\n    FROM workspace.reencontra.ouro_fato_desaparecimentos f\n    JOIN workspace.reencontra.ouro_dim_mes_desaparecimentos d\n        ON f.mes_id = d.mes_id\n    WHERE f.uf = 'SP'\n    GROUP BY d.ano, f.indicador\n    ORDER BY d.ano, f.indicador\n"))


ano,indicador,quantidade_informada,combinacoes_disponiveis,combinacoes_esperadas,cobertura
2023,desaparecimento,18421,180,180,Completo nos campos selecionados
2023,localizacao,16412,180,180,Completo nos campos selecionados
2024,desaparecimento,19966,180,180,Completo nos campos selecionados
2024,localizacao,15969,180,180,Completo nos campos selecionados
2025,desaparecimento,11173,180,180,Completo nos campos selecionados
2025,localizacao,16480,179,180,Incompleto nos campos selecionados


## Consulta 2

Em novembro de 2025, a fonte apresenta 4.071 registros. A diferença entre meses exige investigação da fonte e não demonstra impacto do MVP.


In [0]:
display(spark.sql("\n    SELECT\n        d.ano,\n        d.mes,\n        SUM(f.quantidade) AS registros_desaparecimento,\n        COUNT(*) AS perfis_disponiveis\n    FROM workspace.reencontra.ouro_fato_desaparecimentos f\n    JOIN workspace.reencontra.ouro_dim_mes_desaparecimentos d\n        ON f.mes_id = d.mes_id\n    WHERE f.uf = 'SP'\n      AND f.indicador = 'desaparecimento'\n    GROUP BY d.ano, d.mes\n    ORDER BY d.ano, d.mes\n"))


ano,mes,registros_desaparecimento,perfis_disponiveis
2023,1,1495,15
2023,2,1337,15
2023,3,1600,15
2023,4,1449,15
2023,5,1590,15
2023,6,1358,15
2023,7,1468,15
2023,8,1521,15
2023,9,1548,15
2023,10,1577,15


## Consulta 3

Em 2024, adultos somam 13.036 registros e adolescentes 4.320. Estes são volumes absolutos, não taxas de risco por população.


In [0]:
display(spark.sql("\n    SELECT\n        d.ano,\n        p.faixa_etaria,\n        SUM(f.quantidade) AS registros_desaparecimento\n    FROM workspace.reencontra.ouro_fato_desaparecimentos f\n    JOIN workspace.reencontra.ouro_dim_mes_desaparecimentos d\n        ON f.mes_id = d.mes_id\n    JOIN workspace.reencontra.ouro_dim_perfil_desaparecimentos p\n        ON f.perfil_id = p.perfil_id\n    WHERE f.uf = 'SP'\n      AND f.indicador = 'desaparecimento'\n    GROUP BY d.ano, p.faixa_etaria\n    ORDER BY d.ano, registros_desaparecimento DESC\n"))


ano,faixa_etaria,registros_desaparecimento
2023,adulto,12293
2023,adolescente,3793
2023,idoso,1524
2023,crianca,411
2023,nao_informada,400
2024,adulto,13036
2024,adolescente,4320
2024,idoso,1787
2024,crianca,487
2024,nao_informada,336


## Consulta 4

Em 2024, há 13.174 registros masculinos, 6.774 femininos e 18 com sexo não informado.


In [0]:
display(spark.sql("\n    SELECT\n        d.ano,\n        p.sexo,\n        SUM(f.quantidade) AS registros_desaparecimento\n    FROM workspace.reencontra.ouro_fato_desaparecimentos f\n    JOIN workspace.reencontra.ouro_dim_mes_desaparecimentos d\n        ON f.mes_id = d.mes_id\n    JOIN workspace.reencontra.ouro_dim_perfil_desaparecimentos p\n        ON f.perfil_id = p.perfil_id\n    WHERE f.uf = 'SP'\n      AND f.indicador = 'desaparecimento'\n    GROUP BY d.ano, p.sexo\n    ORDER BY d.ano, registros_desaparecimento DESC\n"))


ano,sexo,registros_desaparecimento
2023,masculino,12393
2023,feminino,6011
2023,nao_informado,17
2024,masculino,13174
2024,feminino,6774
2024,nao_informado,18
2025,masculino,8509
2025,feminino,2664
2025,nao_informado,0


## Consulta 5

De 2023 para 2024, os registros informados passam de 18.421 para 19.966: mais 1.545, variação de 8,39%. Essa é uma comparação descritiva, sem atribuição ao MVP.


In [0]:
display(spark.sql("\n    WITH totais AS (\n        SELECT\n            d.ano,\n            SUM(f.quantidade) AS total\n        FROM workspace.reencontra.ouro_fato_desaparecimentos f\n        JOIN workspace.reencontra.ouro_dim_mes_desaparecimentos d\n            ON f.mes_id = d.mes_id\n        WHERE f.uf = 'SP'\n          AND f.indicador = 'desaparecimento'\n          AND d.ano IN (2023, 2024)\n        GROUP BY d.ano\n    ),\n    comparacao AS (\n        SELECT *,\n            LAG(total) OVER (ORDER BY ano) AS total_ano_anterior\n        FROM totais\n    )\n    SELECT\n        ano,\n        total,\n        total_ano_anterior,\n        total - total_ano_anterior AS variacao_registros,\n        ROUND(\n            100.0 * (total - total_ano_anterior)\n            / NULLIF(total_ano_anterior, 0),\n            2\n        ) AS variacao_percentual\n    FROM comparacao\n    ORDER BY ano\n"))


ano,total,total_ano_anterior,variacao_registros,variacao_percentual
2023,18421,null,null,null
2024,19966,18421,1545,8.39


In [0]:
# Gerar contrato JSON a partir da Ouro; nao utiliza dados de demonstracao
import json
from datetime import datetime, timezone
from decimal import Decimal

consultas_exportacao = {'totais_anuais': "\n    SELECT\n        d.ano,\n        f.indicador,\n        SUM(f.quantidade) AS quantidade_informada,\n        COUNT(*) AS combinacoes_disponiveis,\n        180 AS combinacoes_esperadas,\n        CASE\n            WHEN COUNT(*) = 180 THEN 'Completo nos campos selecionados'\n            ELSE 'Incompleto nos campos selecionados'\n        END AS cobertura\n    FROM workspace.reencontra.ouro_fato_desaparecimentos f\n    JOIN workspace.reencontra.ouro_dim_mes_desaparecimentos d\n        ON f.mes_id = d.mes_id\n    WHERE f.uf = 'SP'\n    GROUP BY d.ano, f.indicador\n    ORDER BY d.ano, f.indicador\n", 'serie_mensal': "\n    SELECT\n        d.ano,\n        d.mes,\n        SUM(f.quantidade) AS registros_desaparecimento,\n        COUNT(*) AS perfis_disponiveis\n    FROM workspace.reencontra.ouro_fato_desaparecimentos f\n    JOIN workspace.reencontra.ouro_dim_mes_desaparecimentos d\n        ON f.mes_id = d.mes_id\n    WHERE f.uf = 'SP'\n      AND f.indicador = 'desaparecimento'\n    GROUP BY d.ano, d.mes\n    ORDER BY d.ano, d.mes\n", 'faixa_etaria': "\n    SELECT\n        d.ano,\n        p.faixa_etaria,\n        SUM(f.quantidade) AS registros_desaparecimento\n    FROM workspace.reencontra.ouro_fato_desaparecimentos f\n    JOIN workspace.reencontra.ouro_dim_mes_desaparecimentos d\n        ON f.mes_id = d.mes_id\n    JOIN workspace.reencontra.ouro_dim_perfil_desaparecimentos p\n        ON f.perfil_id = p.perfil_id\n    WHERE f.uf = 'SP'\n      AND f.indicador = 'desaparecimento'\n    GROUP BY d.ano, p.faixa_etaria\n    ORDER BY d.ano, registros_desaparecimento DESC\n", 'sexo': "\n    SELECT\n        d.ano,\n        p.sexo,\n        SUM(f.quantidade) AS registros_desaparecimento\n    FROM workspace.reencontra.ouro_fato_desaparecimentos f\n    JOIN workspace.reencontra.ouro_dim_mes_desaparecimentos d\n        ON f.mes_id = d.mes_id\n    JOIN workspace.reencontra.ouro_dim_perfil_desaparecimentos p\n        ON f.perfil_id = p.perfil_id\n    WHERE f.uf = 'SP'\n      AND f.indicador = 'desaparecimento'\n    GROUP BY d.ano, p.sexo\n    ORDER BY d.ano, registros_desaparecimento DESC\n", 'variacao_anual': "\n    WITH totais AS (\n        SELECT\n            d.ano,\n            SUM(f.quantidade) AS total\n        FROM workspace.reencontra.ouro_fato_desaparecimentos f\n        JOIN workspace.reencontra.ouro_dim_mes_desaparecimentos d\n            ON f.mes_id = d.mes_id\n        WHERE f.uf = 'SP'\n          AND f.indicador = 'desaparecimento'\n          AND d.ano IN (2023, 2024)\n        GROUP BY d.ano\n    ),\n    comparacao AS (\n        SELECT *,\n            LAG(total) OVER (ORDER BY ano) AS total_ano_anterior\n        FROM totais\n    )\n    SELECT\n        ano,\n        total,\n        total_ano_anterior,\n        total - total_ano_anterior AS variacao_registros,\n        ROUND(\n            100.0 * (total - total_ano_anterior)\n            / NULLIF(total_ano_anterior, 0),\n            2\n        ) AS variacao_percentual\n    FROM comparacao\n    ORDER BY ano\n"}
conteudo = {
    "projeto": "Reencontra",
    "versao_formato": "2.0",
    "uf": "SP",
    "gerado_em": datetime.now(timezone.utc).isoformat(),
    "fonte": "MJSP; dimensao territorial baseada no IBGE",
    "dado_sintetico": False,
    "avisos": [
        "Estatisticas agregadas estaduais de registros informados.",
        "Localizacao em 2025 tem um campo ausente, excluido das somas.",
        "Cobertura de campos nao garante cobertura de todas as ocorrencias.",
        "Os indicadores nao medem impacto do Reencontra nem taxa de resolucao.",
    ],
}
for nome, sql in consultas_exportacao.items():
    conteudo[nome] = [linha.asDict() for linha in spark.sql(sql).collect()]

def converter(valor):
    if isinstance(valor, Decimal):
        return float(valor)
    raise TypeError(f"Tipo nao serializavel: {type(valor)}")

texto = json.dumps(conteudo, ensure_ascii=False, indent=2, default=converter)
caminho = "/Volumes/workspace/reencontra/chegada/saida_mvp/indicadores_sp.json"
dbutils.fs.mkdirs("/Volumes/workspace/reencontra/chegada/saida_mvp")
dbutils.fs.put(caminho, texto, overwrite=True)
assert json.loads(dbutils.fs.head(caminho, 1000000)) == json.loads(texto)
print("JSON real de SP salvo e conferido:", caminho)
print("Quantidade de bytes:", len(texto.encode("utf-8")))


Wrote 9140 bytes.
JSON real de SP salvo e conferido: /Volumes/workspace/reencontra/chegada/saida_mvp/indicadores_sp.json
Quantidade de bytes: 9140


In [0]:
# Resultados reais informados do teste manual do painel.
# Sem nomes ou outros identificadores pessoais.
spark.sql("""
    CREATE OR REPLACE TABLE
    workspace.reencontra.avaliacao_usabilidade_mvp
    USING DELTA AS
    SELECT
        participante,
        'Encontrar o total de desaparecimentos em SP em 2024' AS tarefa,
        19966 AS resposta_encontrada,
        TRUE AS concluiu_sem_ajuda,
        CAST(NULL AS INT) AS tempo_segundos
    FROM VALUES ('P01'), ('P02'), ('P03') AS t(participante)
""")

display(spark.sql("""
    SELECT
        COUNT(*) AS participantes,
        COUNT_IF(
            concluiu_sem_ajuda AND resposta_encontrada = 19966
        ) AS conclusoes_corretas_sem_ajuda,
        ROUND(
            100.0 * COUNT_IF(
                concluiu_sem_ajuda AND resposta_encontrada = 19966
            ) / COUNT(*),
            2
        ) AS taxa_conclusao_percentual
    FROM workspace.reencontra.avaliacao_usabilidade_mvp
"""))

participantes,conclusoes_corretas_sem_ajuda,taxa_conclusao_percentual
3,3,100.00


In [0]:
# Ouro com resultados agregados, sem códigos dos participantes.
spark.sql("""
    CREATE OR REPLACE TABLE
    workspace.reencontra.ouro_avaliacao_usabilidade
    USING DELTA AS
    SELECT
        tarefa,
        COUNT(*) AS participantes,
        COUNT_IF(
            concluiu_sem_ajuda AND resposta_encontrada = 19966
        ) AS conclusoes_corretas_sem_ajuda,
        ROUND(
            100.0 * COUNT_IF(
                concluiu_sem_ajuda AND resposta_encontrada = 19966
            ) / COUNT(*),
            2
        ) AS taxa_conclusao_percentual
    FROM workspace.reencontra.avaliacao_usabilidade_mvp
    GROUP BY tarefa
""")

display(spark.table(
    "workspace.reencontra.ouro_avaliacao_usabilidade"
))

tarefa,participantes,conclusoes_corretas_sem_ajuda,taxa_conclusao_percentual
Encontrar o total de desaparecimentos em SP em 2024,3,3,100.00


In [0]:
# Demonstração de hash usando códigos de participantes do teste.
spark.sql("""
    CREATE OR REPLACE TABLE
    workspace.reencontra.ouro_usabilidade_hash_demo
    USING DELTA AS
    SELECT
        SHA2(CONCAT('reencontra|', participante), 256)
            AS participante_hash,
        tarefa,
        resposta_encontrada,
        concluiu_sem_ajuda
    FROM workspace.reencontra.avaliacao_usabilidade_mvp
""")

display(spark.table(
    "workspace.reencontra.ouro_usabilidade_hash_demo"
))

participante_hash,tarefa,resposta_encontrada,concluiu_sem_ajuda
7fed8a1f1329ebd696f99ec8aadb59d14aec4a299f66f864280d3427d35f9bf7,Encontrar o total de desaparecimentos em SP em 2024,19966,true
4d13407fb3ed7f8462aa132d533b6765c0eb182cfc6d99dd0b3b89d0a4053a38,Encontrar o total de desaparecimentos em SP em 2024,19966,true
a72632012df39a3692e5b6f228107ab51f001306f85a208fbaa4811e81a45526,Encontrar o total de desaparecimentos em SP em 2024,19966,true
